In [2]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from scipy.stats import zscore
from src import Questions
import matplotlib.patches as mpatches
from functools import reduce


plt.rcParams.update({'font.size': 17})

In [3]:
# creating a dataframe of all of the annotator profiles
EXCLUDED_SCORES = ["2_13", "23_", "24_", "25_", "27_"]
questions = Questions().get_questions()
SCORED_ANNOTATOR_DIR = "annotation_artefacts/final_annotator_objects"
os.makedirs(SCORED_ANNOTATOR_DIR, exist_ok=True)

# making a raw scores (both characteristic and downstream) for each annotator
all_tasktype_df_by_model = {}
all_models = [
    "gpt_4o",
    "llama31_70B_Inst",
    "Qwen25_72B_Inst",
    "llama31_8B_Inst",
    "Qwen25_7B_Inst",
    "llama32_1B_Inst",
    "Qwen25_1dot5B_Inst"
]
for presented_model in all_models:
    all_tasktype_df = {}
    task_types = ['mt_and_summarization', 'syntaxes', 'knowledge_and_reasoning', 'semantics', 'qa_extraction']
    for tt in task_types:
        downstream_score_df = []
        for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
            annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
            with open(annotator_filepath, "r") as  ao:
                annotator_object = json.load(ao)
            
            downstream_scores = {}
            for question_key, values in annotator_object["answers"].items():
                if values["answer"] == None or questions[question_key]["question_type"] != tt:
                    continue
                for qid, score_s in values["downstream_scores"][presented_model].items():
                    postfix = questions[question_key]["metrics"][0] # first metric entry is by convention default
                    matching_keys = [k for k in score_s if k.endswith(f"{postfix}_score")]
                    if not matching_keys:
                        raise ValueError(f"No key ends with '{postfix}' for qid {qid}")
                    correct_key = matching_keys[0]
                    downstream_scores[f"{qid}-sentence_type"] = values["sentence_type"]
                    downstream_scores[f"{qid}-sentence_length"] = values["sentence_length"]
                    downstream_scores[f"{qid}-sentence_grammaticality"] = values["grammaticality"]
                    downstream_scores[correct_key] = score_s[correct_key]

            downstream_score_df.append({
                "id" : annotator_object["id"],
                **downstream_scores,
            })
        all_tasktype_df[tt] = pd.DataFrame(downstream_score_df)
    all_tasktype_df_by_model[presented_model] = all_tasktype_df
    

In [4]:

# propmt LENGTH
CHAR_COLUMN = 1
presented_model = ["gpt_4o", "Qwen25_72B_Inst", "llama31_70B_Inst"]
task_corrdf_by_model = {}
for mmm in presented_model:
    task_correlation_df = []
    for tt in task_types:
        taskdf = all_tasktype_df_by_model[mmm][tt].drop(columns=['id'])
        cols = list(taskdf.columns)
        for i in range(0, len(cols), 4):
            chunk = cols[i:i+4]
            task_qidx = chunk[-1].split("-")[-2]
            if not any(task_qidx.startswith(prefix) for prefix in EXCLUDED_SCORES):
                correlation = taskdf[chunk[CHAR_COLUMN]].corr(taskdf[chunk[-1]])
                task_name = "-".join(chunk[-1].split("-")[:-2]).replace("grammatical_classification_","")
                task_correlation_df.append({
                    "Task Name" : task_name,
                    f"Corr. {mmm.split("_")[0]}" : correlation,
                })
    task_correlation_df = pd.DataFrame(task_correlation_df)
    task_corrdf_by_model[mmm] = task_correlation_df

dfs = list(task_corrdf_by_model.values())
merged_df = reduce(lambda left, right: pd.merge(left, right, on="Task Name", how="outer"), dfs)
merged_df_reformatted = merged_df.applymap(
    lambda x: (
        f"\\textbf{{{x:.2f}}}".replace('.', ',') if isinstance(x, float) and abs(x) > 0.2
        else f"{x:.2f}".replace('.', ',') if isinstance(x, float)
        else x.replace('_', r'\_') if isinstance(x, str)
        else x
    )
)
print(merged_df_reformatted.to_latex(index=False))

\begin{tabular}{llll}
\toprule
Task Name & Corr. gpt & Corr. Qwen25 & Corr. llama31 \\
\midrule
Dialogue Summarization & \textbf{0,45} & \textbf{0,42} & \textbf{0,46} \\
Extractive QA & \textbf{-0,21} & 0,01 & 0,04 \\
Free Generation: Adhan & 0,08 & \textbf{0,21} & -0,10 \\
Free Generation: Arithmetic & 0,05 & 0,02 & -0,12 \\
Free Generation: Balonku Song & -0,02 & 0,16 & 0,17 \\
Free Generation: G30SPKI & 0,10 & 0,04 & 0,12 \\
Free Generation: Giant Drum & \textbf{-0,37} & \textbf{-0,60} & \textbf{-0,40} \\
Free Generation: Grammar Correction I & 0,03 & 0,08 & 0,05 \\
Free Generation: Grammar Correction II & -0,13 & -0,05 & -0,12 \\
Free Generation: Hijrah & 0,04 & -0,09 & 0,09 \\
Free Generation: Hillary Clinton & -0,06 & \textbf{-0,24} & -0,10 \\
Free Generation: Karak Kaliang & \textbf{0,32} & \textbf{0,62} & \textbf{0,29} \\
Free Generation: MBZUAI & 0,10 & -0,04 & 0,15 \\
Free Generation: Olympics Medallists & \textbf{0,21} & 0,14 & 0,02 \\
Free Generation: School Life & 0,06 & -

/var/folders/9t/9r8_fqln16b43khrkh9lb4kh0000gn/T/ipykernel_35788/1329683824.py:25: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  merged_df_reformatted = merged_df.applymap(


In [5]:
def df_with_hexcolor_to_latex(values_df, color_df):
    from collections import OrderedDict

    latex_rows = []
    defined_colors = OrderedDict()
    color_idx = 0

    for i in range(values_df.shape[0]):
        row = []
        for j in range(values_df.shape[1]):
            val = values_df.iat[i, j]
            hex_color = color_df.iat[i, j]
            if isinstance(hex_color, str) and hex_color.startswith("#"):
                if hex_color not in defined_colors:
                    defined_colors[hex_color] = f"customcolor{color_idx}"
                    color_idx += 1
                colorname = defined_colors[hex_color]
                cell = f"\\cellcolor{{{colorname}!30}}{val}"
            else:
                cell = str(val)
            row.append(cell)
        latex_rows.append(" & ".join(row) + r" \\")
    
    col_headers = " & ".join(values_df.columns) + r" \\"
    header = r"\begin{tabular}{" + "l" * values_df.shape[1] + "}\n\\hline\n" + col_headers + "\n\\hline"
    body = "\n".join(latex_rows)
    footer = "\n\\hline\n\\end{tabular}"

    color_defs = "\n".join(
        [f"\\definecolor{{{name}}}{{HTML}}{{{hex[1:]}}}" for hex, name in defined_colors.items()]
    )

    return color_defs + "\n\n" + header + "\n" + body + footer

In [35]:
# ANOVA with bonferroni correction
from scipy.stats import f_oneway

REJECT_THRESHOLD = 0.05

def compute_anova_score(data_dict):
    # Extract the lists of scores
    groups = list(data_dict.values())
    # Perform one-way ANOVA
    f_statistic, p_value = f_oneway(*groups)
    return f_statistic, p_value

# 0 for TONE, 2 for GRAMMAR
CHAR_COLUMN = 0
task_lingcat_scores_by_model = {}
for mmm in all_tasktype_df_by_model.keys():
    task_lingcat_scores_by_model[mmm] = {}
    for tt in task_types:
        task_lingcat_scores_by_model[mmm][tt] = {}
        taskdf = all_tasktype_df_by_model[mmm][tt].drop(columns=['id'])
        cols = list(taskdf.columns)
        for i in range(0, len(cols), 4):
            chunk = cols[i:i+4]
            task_qidx = chunk[-1].split("-")[-2]
            if not any(task_qidx.startswith(prefix) for prefix in EXCLUDED_SCORES):
                task_lingcat_scores_by_model[mmm][tt][task_qidx] = {}
                chunkdf = taskdf[chunk].dropna()
                for ptone, tone_group in chunkdf.groupby(chunk[CHAR_COLUMN]):
                    task_lingcat_scores_by_model[mmm][tt][task_qidx][ptone] = tone_group[chunk[-1]].to_list()

for modl in task_lingcat_scores_by_model.keys():
    for tg in task_lingcat_scores_by_model[modl].keys():
        task_count_in_tg = len(task_lingcat_scores_by_model[modl][tg].keys())
        for qdi in task_lingcat_scores_by_model[modl][tg].keys():
            f, p = compute_anova_score(task_lingcat_scores_by_model[modl][tg][qdi])
            adjusted_p = p * task_count_in_tg * 3
            if (adjusted_p < REJECT_THRESHOLD):
                print(modl, "REJECTED!", f"task_group-{tg}", f"task-{qdi}", adjusted_p)

gpt_4o REJECTED! task_group-mt_and_summarization task-2_9 7.699559595989646e-07
gpt_4o REJECTED! task_group-mt_and_summarization task-2_8 7.876783319428735e-10
gpt_4o REJECTED! task_group-mt_and_summarization task-2_7 0.0001273894730503381
gpt_4o REJECTED! task_group-mt_and_summarization task-2_6 1.1774553428889193e-06
gpt_4o REJECTED! task_group-mt_and_summarization task-2_3 2.5000147892881943e-05
gpt_4o REJECTED! task_group-mt_and_summarization task-2_2 4.069511142324739e-09
gpt_4o REJECTED! task_group-mt_and_summarization task-2_1 6.298367507299101e-05
gpt_4o REJECTED! task_group-mt_and_summarization task-2_5 0.0005085411910092163
Qwen25_72B_Inst REJECTED! task_group-mt_and_summarization task-2_8 0.044373659518878325
Qwen25_72B_Inst REJECTED! task_group-mt_and_summarization task-2_3 0.0007291920636991866
Qwen25_72B_Inst REJECTED! task_group-mt_and_summarization task-2_1 0.0017618877490495838
Qwen25_72B_Inst REJECTED! task_group-knowledge_and_reasoning task-12_0 8.19731270826418e-05


/Users/rayendito/miniconda3/envs/pietorch/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:573: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, **kwds)


In [ ]:
# naive p values

from scipy.stats import ttest_ind
from itertools import combinations

# 0 for TONE, 2 for GRAMMAR
CHAR_COLUMN = 0
task_lingcat_scores_by_model = {}
for mmm in all_tasktype_df_by_model.keys():
    task_lingcat_scores_by_model[mmm] = {}
    for tt in task_types:
        task_lingcat_scores_by_model[mmm][tt] = {}
        taskdf = all_tasktype_df_by_model[mmm][tt].drop(columns=['id'])
        cols = list(taskdf.columns)
        for i in range(0, len(cols), 4):
            chunk = cols[i:i+4]
            task_qidx = chunk[-1].split("-")[-2]
            if not any(task_qidx.startswith(prefix) for prefix in EXCLUDED_SCORES):
                task_lingcat_scores_by_model[mmm][tt][task_qidx] = {}
                chunkdf = taskdf[chunk].dropna()
                for ptone, tone_group in chunkdf.groupby(chunk[CHAR_COLUMN]):
                    task_lingcat_scores_by_model[mmm][tt][task_qidx][ptone] = tone_group[chunk[-1]].to_list()

REJECT_THRESHOLD = 0.05
n_Ho_rejects_totalreject_df = []
n_Ho_rejects_which_df = []

color_dict = {
    "imperative" : "#1f77b4",
    "interrogative" : "#ff7f0e",
    "declarative" : "#2ca02c",
    "correct" : "#6084f7",
    "incorrect" : "#d6425d",
    "white" : "#ffffff",
}

def pairwise_p_values_with_higher(data_dict):
    results = {}
    for (key1, values1), (key2, values2) in combinations(data_dict.items(), 2):
        _, p = ttest_ind(values1, values2, equal_var=False)
        
        mean1 = sum(values1) / len(values1)
        mean2 = sum(values2) / len(values2)
        higher = key1 if mean1 > mean2 else key2

        results[f"{key1} ---- {key2}"] = {
            "p" : p,
            "higher" : higher,
        }
    return results

n_Ho_rejects_data = {}
for mmm in task_lingcat_scores_by_model.keys():
    n_Ho_rejects_data[mmm] = {}
    for tg in task_lingcat_scores_by_model[mmm].keys():
        n_Ho_rejects_data[mmm][tg] = {}
        n_Ho_rejects_data[mmm][tg]["pairwise_unfair"] = {}
        unfair_tasks = []
        for qidx in task_lingcat_scores_by_model[mmm][tg].keys():
            this_task_unfair = False
            ttest_result = pairwise_p_values_with_higher(task_lingcat_scores_by_model[mmm][tg][qidx])
            for ttest_values in ttest_result.values():
                if ttest_values['p'] < REJECT_THRESHOLD:
                    this_task_unfair = True
                    if ttest_values['higher'] in n_Ho_rejects_data[mmm][tg]["pairwise_unfair"]:
                        n_Ho_rejects_data[mmm][tg]["pairwise_unfair"][ttest_values['higher']] += 1
                    else:
                        n_Ho_rejects_data[mmm][tg]["pairwise_unfair"][ttest_values['higher']] = 1
            unfair_tasks.append(this_task_unfair)
        n_Ho_rejects_data[mmm][tg]["unfair_task_percentage"] = sum(unfair_tasks)/len(unfair_tasks)
            
for model in n_Ho_rejects_data:
    tg_values_total = {}
    tg_values_which = {}
    for tg, values in n_Ho_rejects_data[model].items():
        tg_values_total[tg] = f"{round(values['unfair_task_percentage'] * 100, 1)}\%"
        tg_values_which[tg] = color_dict[max(values["pairwise_unfair"], key=values["pairwise_unfair"].get) if values["pairwise_unfair"] else "white"]

    n_Ho_rejects_totalreject_df.append({
        "Model" : model,
        **tg_values_total,
    })

    n_Ho_rejects_which_df.append({
        "Model" : model,
        **tg_values_which,
    })


n_Ho_rejects_totalreject_df = pd.DataFrame(n_Ho_rejects_totalreject_df)
n_Ho_rejects_which_df = pd.DataFrame(n_Ho_rejects_which_df)

<>:74: SyntaxWarning: invalid escape sequence '\%'
<>:74: SyntaxWarning: invalid escape sequence '\%'
/var/folders/9t/9r8_fqln16b43khrkh9lb4kh0000gn/T/ipykernel_35788/2882532141.py:74: SyntaxWarning: invalid escape sequence '\%'
  tg_values_total[tg] = f"{round(values['unfair_task_percentage'] * 100, 1)}\%"
/Users/rayendito/miniconda3/envs/pietorch/lib/python3.12/site-packages/scipy/stats/_stats_py.py:1114: RuntimeWarning: divide by zero encountered in divide
  var *= np.divide(n, n-ddof)  # to avoid error on division by zero
/Users/rayendito/miniconda3/envs/pietorch/lib/python3.12/site-packages/scipy/stats/_stats_py.py:1114: RuntimeWarning: invalid value encountered in scalar multiply
  var *= np.divide(n, n-ddof)  # to avoid error on division by zero
/Users/rayendito/miniconda3/envs/pietorch/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:573: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nea

In [10]:
n_Ho_rejects_totalreject_df

,Model,mt_and_summarization,syntaxes,knowledge_and_reasoning,semantics,qa_extraction
0,gpt_4o,80.0\%,0.0\%,23.1\%,33.3\%,0.0\%
1,llama31_70B_Inst,26.7\%,20.0\%,23.1\%,33.3\%,33.3\%
2,Qwen25_72B_Inst,53.3\%,40.0\%,38.5\%,33.3\%,0.0\%
3,llama31_8B_Inst,60.0\%,0.0\%,15.4\%,100.0\%,0.0\%
4,Qwen25_7B_Inst,40.0\%,20.0\%,0.0\%,33.3\%,33.3\%
5,llama32_1B_Inst,66.7\%,0.0\%,15.4\%,0.0\%,33.3\%
6,Qwen25_1dot5B_Inst,26.7\%,20.0\%,7.7\%,0.0\%,33.3\%


In [ ]:
n_Ho_rejects_which_df

,Model,mt_and_summarization,syntaxes,knowledge_and_reasoning,semantics,qa_extraction
0,gpt_4o,#1f77b4,#ffffff,#ff7f0e,#1f77b4,#ffffff
1,llama31_70B_Inst,#2ca02c,#ff7f0e,#2ca02c,#ff7f0e,#ff7f0e
2,Qwen25_72B_Inst,#1f77b4,#2ca02c,#1f77b4,#1f77b4,#ffffff
3,llama31_8B_Inst,#ff7f0e,#ffffff,#ff7f0e,#ff7f0e,#ffffff
4,Qwen25_7B_Inst,#1f77b4,#2ca02c,#ffffff,#2ca02c,#2ca02c
5,llama32_1B_Inst,#ff7f0e,#ffffff,#1f77b4,#ffffff,#1f77b4
6,Qwen25_1dot5B_Inst,#1f77b4,#2ca02c,#ff7f0e,#ffffff,#1f77b4


In [ ]:
print(df_with_hexcolor_to_latex(n_Ho_rejects_totalreject_df, n_Ho_rejects_which_df))

\definecolor{customcolor0}{HTML}{1f77b4}
\definecolor{customcolor1}{HTML}{ffffff}
\definecolor{customcolor2}{HTML}{ff7f0e}
\definecolor{customcolor3}{HTML}{2ca02c}

\begin{tabular}{llllll}
\hline
Model & mt_and_summarization & syntaxes & knowledge_and_reasoning & semantics & qa_extraction \\
\hline
gpt_4o & \cellcolor{customcolor0!30}80.0\% & \cellcolor{customcolor1!30}0.0\% & \cellcolor{customcolor2!30}23.1\% & \cellcolor{customcolor0!30}33.3\% & \cellcolor{customcolor1!30}0.0\% \\
llama31_70B_Inst & \cellcolor{customcolor3!30}26.7\% & \cellcolor{customcolor2!30}20.0\% & \cellcolor{customcolor3!30}23.1\% & \cellcolor{customcolor2!30}33.3\% & \cellcolor{customcolor2!30}33.3\% \\
Qwen25_72B_Inst & \cellcolor{customcolor0!30}53.3\% & \cellcolor{customcolor3!30}40.0\% & \cellcolor{customcolor0!30}38.5\% & \cellcolor{customcolor0!30}33.3\% & \cellcolor{customcolor1!30}0.0\% \\
llama31_8B_Inst & \cellcolor{customcolor2!30}60.0\% & \cellcolor{customcolor1!30}0.0\% & \cellcolor{customcolor2!3